# Platform Integrations Functionality Test

Tests the newer platform/infrastructure helpers:

- token budget guard (`core.llm_guard`)
- retry helper (`core.retry`)
- pgvector facade (`core.vector_store`)
- MCP client fallback (`core.mcp_client`)
- `VectorDBConfig` in settings

In [ ]:
from pathlib import Path
import os
import sys

cwd = Path.cwd().resolve()
project_root = next((p for p in [cwd, *cwd.parents] if (p / 'src').exists()), cwd)
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print('project_root:', project_root)

## 1. Token Budget Guard

In [ ]:
from core.llm_guard import DAILY_TOKEN_LIMIT, check_and_record_tokens, estimate_tokens, get_daily_usage

print('DAILY_TOKEN_LIMIT:', DAILY_TOKEN_LIMIT)
print('estimate_tokens(400 chars):', estimate_tokens('a' * 400))
assert estimate_tokens('a' * 400) == 400 // 4 + 500
assert check_and_record_tokens(None, 1000) is True
print('usage without redis:', get_daily_usage(None))

In [ ]:
class FakeRedis:
    def __init__(self):
        self.values = {}
        self.expiries = {}
    def incrby(self, key, amount):
        self.values[key] = self.values.get(key, 0) + amount
        return self.values[key]
    def expire(self, key, seconds):
        self.expiries[key] = seconds
    def get(self, key):
        return self.values.get(key, 0)

r = FakeRedis()
assert check_and_record_tokens(r, 100) is True
usage = get_daily_usage(r)
print('usage:', usage)
assert usage['tokens_used'] == 100
assert usage['redis_ok'] is True

## 2. Retry Helper

In [ ]:
import core.retry as retry
from core.retry import retry_agent_call, with_retry

retry.time.sleep = lambda _: None

calls = {'n': 0}
@with_retry(max_retries=3, backoff_factor=0.01)
def flaky():
    calls['n'] += 1
    if calls['n'] < 2:
        raise ConnectionError('temporary')
    return 'ok'

assert flaky() == 'ok'
assert calls['n'] == 2
print('decorator retry calls:', calls['n'])

def always_fail(req):
    raise RuntimeError('down')

result = retry_agent_call(always_fail, object(), max_retries=2)
print(result.to_dict())
assert result.success is False
assert 'Failed after' in result.error

## 3. VectorDBConfig and Vector Store Fallback

In [ ]:
from config.settings import AppConfig, VectorDBConfig
from core.vector_store import get_vector_store, similarity_search

cfg = AppConfig().vector_db
print('vector config:', cfg)
print('connection string redacted:', cfg.connection_string.replace(cfg.password, '<redacted>') if cfg.password else cfg.connection_string)

store = get_vector_store(cfg)
print('store:', type(store))
results = similarity_search(store, 'What is GRR for retention?', k=3)
for doc, score in results:
    print(score, doc.metadata, doc.page_content[:120])
assert results
assert all(0 <= score <= 1 for _, score in results)

## 4. MCP Client Fallback

In [ ]:
from core.mcp_client import get_mcp_tools, is_mcp_enabled, list_configured_servers

print('USE_MCP enabled:', is_mcp_enabled())
print('configured servers:', list_configured_servers())
print('collibra tools:', get_mcp_tools('collibra'))
print('jira tools:', get_mcp_tools('jira'))

if not is_mcp_enabled():
    assert get_mcp_tools('collibra') == []
    assert get_mcp_tools('jira') == []